In [11]:
from dotenv import load_dotenv

# import Chatgroq , tools , create_agents
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

PRODUCTS = {
    "wireless headphones" : {"price" : 1499, "description": "Wireless headphones delivers music and calls through Bluetooth, freeing you from tangled wires. seamless pairing with smartphones, laptops, and other devices without cables. ranging from 40 hours. high-fidelity audio, noise cancellation, or ambient sound modes."},
    "smart watch" : {"price" : 1999 , "description" : "connects via Bluetooth or Wi-Fi to track health, deliver notifications, and enable smart features without cables. Monitors heart rate, steps, sleep, and workouts. Syncs with smartphones for calls, messages, and app alerts. Offers long-lasting power with water and dust resistance for daily use."},
    "mechanical keyboard": {"price" : 1299 , "description" : "individual mechanical switches for each key, delivering precision and durability.  Offers distinct switch types (linear, tactile, clicky) for customized typing feel.  last tens of millions of keystrokes, Supports features like keycap replacement, RGB lighting, and programmable macros."},
    "laptop stand" : {"price": 499 , "description" : "better posture, cooling, and productivity. customization for comfortable viewing and typing. made of metal,  Lightweight and foldable."}
}

REVIEW = {
    "wireless headphones" : {"review" : 1576 , "rating" : 4.3} ,
    "smart watch" : {"review" : 1474 , "rating" : 4.7},
    "mechanical keyboard": {"review" : 4566 , "rating" : 4.1},
    "laptop stand" : {"review" : 2374 , "rating" : 4.2}
}


@tool
def get_product(name : str):
    """ Look up a product by its name and return its price, rating, stock and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return "Product is not available. "
    return str(p)

@tool
def get_review(name: str):
    """ Look up a product review by its name and returns products rating and their review."""
    r = REVIEW.get(name.lower())
    if not r:
        return "Product is not available, so no review found."
    return str(r)

llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature = 0.5
)

agent  = create_agent(
    llm,
    tools = [get_product , get_review],
    system_prompt = "you are helpful product assistant for an online store.",
    checkpointer = InMemorySaver()
)

def ask(question : str):
    config = {"configurable" : {"thread_id" :"user_session1"}}
    result = agent.invoke({"messages" :[{"role" : "user" , "content":question}]},
    config = config)
    print(result["messages"][-1].content)

    

In [12]:
ask("What is the price of wireless headphones")

The price of the wireless headphones is $1499. They have a description of delivering music and calls through Bluetooth, with seamless pairing and up to 40 hours of battery life, and features such as high-fidelity audio, noise cancellation, or ambient sound modes.


In [18]:
def ask2(question : str):
    config = {"configurable" : {"thread_id" : "user_session1"}}
    result = agent.invoke({"messages" : [{"role" : "user" , "content" : question}]},
    config = config)
    print(result["messages"][-1].content)

In [19]:
ask2("what is the review of this prodcut")

The review of the wireless headphones has a rating of 4.3 and the review is: "Great sound quality, comfortable to wear and long battery life. The noise cancellation feature is also very effective. However, the price is a bit steep for some users."
